In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/nlp-getting-started/sample_submission.csv
/kaggle/input/competitions/nlp-getting-started/train.csv
/kaggle/input/competitions/nlp-getting-started/test.csv


In [2]:
import re
import numpy as np
import pandas as pd

from scipy.sparse import hstack

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [3]:
train = pd.read_csv("/kaggle/input/competitions/nlp-getting-started/train.csv")
test = pd.read_csv("/kaggle/input/competitions/nlp-getting-started/test.csv")
sample_sub = pd.read_csv("/kaggle/input/competitions/nlp-getting-started/sample_submission.csv")

In [4]:
def clean_text(text):
    text = str(text).lower()

    text = re.sub(r'https?://\S+|www\.\S+', ' URL ', text)
    text = re.sub(r'@\w+', ' USER ', text)
    text = re.sub(r'#(\w+)', r' \1 ', text)

    text = re.sub(r"can't", "cannot", text)
    text = re.sub(r"won't", "will not", text)
    text = re.sub(r"n't", " not", text)
    text = re.sub(r"'re", " are", text)
    text = re.sub(r"'ll", " will", text)
    text = re.sub(r"'ve", " have", text)

    text = re.sub(r'[^a-z0-9 ]', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [5]:
for df in (train, test):
    df['keyword'] = df['keyword'].fillna('')

    df['text_clean'] = df['text'].apply(clean_text)

    df['keyword_clean'] = df['keyword'].str.replace('%20', ' ', regex=False)

    df['combined'] = (
        df['keyword_clean'] + ' ' +
        df['text_clean']
    ).str.strip()

In [6]:
word_vec = TfidfVectorizer(
    analyzer="word",
    lowercase=True,
    stop_words="english",
    strip_accents="unicode",
    ngram_range=(1,2),
    min_df=1,
    max_df=0.95,
    sublinear_tf=True,
    max_features=60000
)

char_vec = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3,6),
    min_df=2,
    sublinear_tf=True,
    max_features=50000
)

In [7]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

svm = LinearSVC(
    C=1,
    max_iter=10000
)

scores = cross_val_score(
    svm,
    X_train,
    y,
    cv=skf,
    scoring='f1'
)

print("F1 Scores:", scores)
print("Mean F1:", scores.mean())

NameError: name 'X_train' is not defined

In [ ]:
final_model = CalibratedClassifierCV(
    LinearSVC(
        C=1,
        max_iter=10000
    ),
    cv=5
)

final_model.fit(X_train, y)

predictions = final_model.predict(X_test)

In [ ]:
submission = pd.DataFrame({
    'id': test['id'],
    'target': predictions.astype(int)
})

submission = submission.set_index('id').loc[sample_sub['id']].reset_index()

submission.to_csv("submission.csv", index=False)

print("Submission file created successfully!")
print(submission.head())
submission.shape

In [ ]:
import os

os.listdir("/kaggle/working")